# Quantum Golden Pendulum Chaos Engine

**Hybrid quantum-classical experiment on IBM Marrakesh 156-qubit Heron r2**

This notebook demonstrates that anti-resonant (maximally irrational) weight sequences
stabilize quantum simulation of coupled-oscillator Hamiltonians better than rational baselines.

## Setup
1. Install: `pip install qiskit qiskit-ibm-runtime qiskit-aer matplotlib`
2. For real hardware: save your IBM Quantum token first
3. This notebook uses FakeMarrakesh by default (no QPU cost)

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

from quantum_golden_pendulum.anti_resonant_weights import (
    get_weights, rational_baseline_weights, weights_to_angles,
    WEIGHT_MODES, PHI, TWO_PHI,
)
from quantum_golden_pendulum.hamiltonian import build_pendulum_hamiltonian
from quantum_golden_pendulum.calibration import pull_calibration, select_qubit_subset
from quantum_golden_pendulum.ansatz import build_pendulum_ansatz
from quantum_golden_pendulum.runtime_job import RuntimeManager, get_fake_backend
from quantum_golden_pendulum.optimizer import QuantumGoldenPendulumOptimizer
from quantum_golden_pendulum.conserved import measure_conserved_quantities
from quantum_golden_pendulum.plotting import (
    plot_energy_convergence, plot_phase_convergence,
    plot_conserved_quantities, plot_comparison_bars,
)

print('All imports successful!')

## 1. Visualize Anti-Resonant Weight Families

In [ ]:
# Compare weight distributions across all modes
N = 20  # Number of oscillators
modes = ['golden', 'bronze', 'cocktail', 'chaotic_logistic', 'silver', 'plastic']

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for idx, mode in enumerate(modes):
    ax = axes[idx // 3][idx % 3]
    w = get_weights(mode, N)
    ax.bar(range(N), w, color=plt.cm.viridis(w / w.max()))
    ax.set_title(f'{mode} (max/min = {w.max()/w.min():.1f}x)', fontweight='bold')
    ax.set_ylabel('Weight')
    ax.set_xlabel('Oscillator index')

fig.suptitle('Anti-Resonant Weight Families (K=20)', fontsize=14, fontweight='bold')
fig.tight_layout()
plt.show()

# Show the rotation angles on the Bloch sphere
fig, ax = plt.subplots(figsize=(10, 4))
for mode in ['golden', 'chaotic_logistic', 'uniform']:
    if mode == 'uniform':
        w = rational_baseline_weights(N, mode='uniform')
    else:
        w = get_weights(mode, N)
    angles = weights_to_angles(w)
    ax.stem(range(N), angles / np.pi, label=mode, markerfmt='o', basefmt=' ')

ax.set_xlabel('Qubit index')
ax.set_ylabel('Fixed angle / pi')
ax.set_title('Layer 0 Fixed Rotation Angles', fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 2. Build the Hamiltonian and Ansatz

In [ ]:
# Setup: FakeMarrakesh backend
backend = get_fake_backend()
print(f'Backend: {backend.name}, {backend.num_qubits} qubits')

# Pull calibration
cal = pull_calibration(backend)
print(f'Good qubits: {len(cal.good_qubits)}, Bad: {len(cal.bad_qubits)}, Dead: {len(cal.dead_qubits)}')

# Select a manageable subset for the demo
N_QUBITS = 16  # Use 16 for quick demo; scale to 156 for real runs
selected_qubits, selected_edges = select_qubit_subset(cal, N_QUBITS)
qubit_to_local = {q: i for i, q in enumerate(selected_qubits)}
local_edges = [(qubit_to_local[i], qubit_to_local[j])
               for (i, j) in selected_edges
               if i in qubit_to_local and j in qubit_to_local]

print(f'Selected {N_QUBITS} qubits with {len(local_edges)} edges')
print(f'Physical qubits: {selected_qubits}')

In [ ]:
# Build Hamiltonian with golden weights
weights_golden = get_weights('golden', N_QUBITS)
H = build_pendulum_hamiltonian(
    n_qubits=N_QUBITS,
    alpha_weights=weights_golden,
    coupling_pairs=local_edges,
)
print(f'Hamiltonian: {len(H)} Pauli terms')
print(f'Top 5 terms by |coefficient|:')
from quantum_golden_pendulum.hamiltonian import decompose_for_measurement
for pauli_str, coeff in decompose_for_measurement(H)[:5]:
    print(f'  {coeff:+.6f} * {pauli_str}')

In [ ]:
# Build ansatz circuit
angles_golden = weights_to_angles(weights_golden)
ansatz, params = build_pendulum_ansatz(
    n_qubits=N_QUBITS,
    fixed_angles=angles_golden,
    coupling_edges=local_edges,
    n_var_layers=2,  # 2 layers for demo
)
print(f'Ansatz: {N_QUBITS} qubits, {len(params)} trainable params, depth {ansatz.depth()}')
ansatz.draw('mpl', fold=80)

## 3. Run the Comparative Experiment

Compare 4 anti-resonant modes + 2 rational baselines.

In [ ]:
# Run optimization for each mode
runtime = RuntimeManager(backend=backend, optimization_level=2)

MODES = ['golden', 'bronze', 'cocktail', 'chaotic_logistic']
BASELINES = ['uniform', 'harmonic']
MAX_ITER = 20  # Increase to 50-100 for real experiments
N_VAR_LAYERS = 2

all_results = {}

for mode in MODES + [f'baseline_{b}' for b in BASELINES]:
    print(f'\n--- Running: {mode} ---')
    
    if mode.startswith('baseline_'):
        w = rational_baseline_weights(N_QUBITS, mode=mode.replace('baseline_', ''))
    else:
        w = get_weights(mode, N_QUBITS)
    
    angles = weights_to_angles(w)
    circuit, _ = build_pendulum_ansatz(
        n_qubits=N_QUBITS,
        fixed_angles=angles,
        coupling_edges=local_edges,
        n_var_layers=N_VAR_LAYERS,
    )
    
    H_mode = build_pendulum_hamiltonian(
        n_qubits=N_QUBITS,
        alpha_weights=w,
        coupling_pairs=local_edges,
    )
    
    optimizer = QuantumGoldenPendulumOptimizer(
        runtime_manager=runtime,
        ansatz_circuit=circuit,
        hamiltonian=H_mode,
        n_qubits=N_QUBITS,
        weight_mode=mode,
        n_var_layers=N_VAR_LAYERS,
        max_iterations=MAX_ITER,
        shots=2000,
    )
    
    result = optimizer.run()
    all_results[mode] = result
    print(f'  Best energy: {result.best_energy:+.6f} at step {result.best_step}')

print('\nAll modes complete!')

## 4. Results Visualization

In [ ]:
plot_energy_convergence(all_results, title=f'Energy Convergence ({N_QUBITS} qubits, FakeMarrakesh)')

In [ ]:
plot_phase_convergence(all_results)

In [ ]:
plot_conserved_quantities(all_results)

In [ ]:
plot_comparison_bars(all_results)

## 5. Conservation Law Analysis

Do the wave conservation laws (E1=1, L2=pi, L4=sqrt(e)) hold for quantum data?

In [ ]:
# Print final conservation law values for each mode
print('CONSERVATION LAW FINAL VALUES')
print('=' * 70)
for mode, result in sorted(all_results.items()):
    if result.steps:
        final = result.steps[-1].conserved
        print(f'\n{mode}:')
        print(final.summary())

## 6. Scale to Full 156 Qubits (Real Hardware)

Uncomment and modify the cell below to run on the real IBM Marrakesh processor.

In [ ]:
# # REAL HARDWARE RUN (requires IBM Quantum token)
# from quantum_golden_pendulum.runtime_job import connect_ibm_service
#
# real_backend = connect_ibm_service(backend_name='ibm_marrakesh')
# real_cal = pull_calibration(real_backend, save_path=Path('../results/calibration.json'))
# real_qubits, real_edges = select_qubit_subset(real_cal, 156)
# real_runtime = RuntimeManager(backend=real_backend)
#
# # Run golden mode on full chip
# w_156 = get_weights('golden', 156)
# angles_156 = weights_to_angles(w_156)
# qubit_map = {q: i for i, q in enumerate(real_qubits)}
# edges_156 = [(qubit_map[i], qubit_map[j]) for i, j in real_edges
#              if i in qubit_map and j in qubit_map]
#
# circuit_156, _ = build_pendulum_ansatz(156, angles_156, edges_156, n_var_layers=3)
# H_156 = build_pendulum_hamiltonian(156, w_156, edges_156)
#
# opt_156 = QuantumGoldenPendulumOptimizer(
#     runtime_manager=real_runtime,
#     ansatz_circuit=circuit_156,
#     hamiltonian=H_156,
#     n_qubits=156,
#     max_iterations=50,
# )
# result_156 = opt_156.run()
# print(f'156-qubit best energy: {result_156.best_energy:+.6f}')